In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
order_items_bronze_path = f"{BRONZE_PATH}/order_items"
order_items_silver_path = f"{SILVER_PATH}/order_items"

In [0]:
df_order_items_bronze = spark.read.format("delta") \
    .load(order_items_bronze_path)

In [0]:
display(df_order_items_bronze.limit(20))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
windowSpec = Window.partitionBy("order_item_id").orderBy(F.col("updated_at").desc())

df_order_items_silver = df_order_items_bronze \
    .withColumn(
        "rn",
        F.row_number().over(windowSpec)
    ) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

In [0]:
df_order_items_silver.write.format("delta").mode("append").save(order_items_silver_path)

In [0]:
from delta.tables import DeltaTable

order_items_silver_table = DeltaTable.forPath(
    spark,
    order_items_silver_path
)

order_items_silver_table.alias("target") \
    .merge(
        df_order_items_silver.alias("source"),
        "source.order_item_id = target.order_item_id"
    ) \
    .whenMatchedUpdate(
        set = {
            "order_id": "source.order_id",
            "product_id": "source.product_id",
            "quantity": "source.quantity",
            "unit_price": "source.unit_price",
            "line_total": "source.line_total",
            "updated_at": "source.updated_at"
        }
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
display(
    spark.read.format("delta") \
        .load(order_items_silver_path)
)